# Module 7 — Code Along: Unit Testing (pytest basics)

> **Run this via `uv run jupyter lab`** (or any kernel with the project deps) — the cells import
> `pydantic` and `pytest`, already in the project's dev dependencies. A bare Python kernel
> without them fails from §1.3 on.

**Why we're here:** you've built a `Product`, a `ProductCatalog`, storage, a server. Change one
line tomorrow and you'd have to re-check every behavior by hand. A **test** is that check,
saved — it re-runs in seconds and tells you instantly if you broke something. That's the whole
payoff: *changing code tomorrow without fear.*

Today everything runs **for real** — a real `Product`-like model, a real `Catalog`-like
collection, a real temp file — because all of it is cheap to run. So there are **no mocks** in
M07. (Mocks earn their keep at the one edge you *can't* run for real — the network client — and
that's M08's job, not this notebook's.)

**Self-contained by design:** every domain class here is a tiny inline stand-in (`Product`,
`Catalog`) so the notebook needs no `catalog` package on the path — only `pydantic` + `pytest`.
In the lab you'll write the same kind of tests against the real `catalog.models.Product` /
`ProductCatalog` and `catalog.storage.save_json` / `load_json`.

Section by section, one code cell per beat — run top to bottom:

**Section 1** `assert` · **Section 2** `pytest.raises` + `@parametrize` · **Section 3** fixtures + `tmp_path`

## Section 1 — Test functions + `assert`

A test is just a function named `test_*` that makes an `assert`. Under `pytest -q` it's
discovered automatically and, on failure, pytest shows **both sides** of the comparison it
rewrote for you — no `assertEqual`, no boilerplate. Here we call each `test_*` function
directly (a notebook isn't a pytest run), so a clean pass just falls through with no output,
and we print a line to make that visible.

In [ ]:
# 1.1 · The smallest possible test — a function named test_*, a plain assert.
def test_math():
    assert 1 + 1 == 2

test_math()   # no exception raised = the assert held
print("test_math: passed")

In [ ]:
# 1.2 · Break it — deliberately wrong, caught so the notebook keeps running.
# Plain Python only raises AssertionError with no message; run under `pytest -q` this same
# line is rewritten to show BOTH sides: "assert 2 == 3". That rewrite is why pytest > unittest.
def test_math_broken():
    assert 1 + 1 == 3

try:
    test_math_broken()
except AssertionError as e:
    print(f"caught AssertionError as pytest would report FAILED (pytest would show: assert 2 == 3): {e!r}")

In [ ]:
# 1.3 · A real object — a tiny Product-like model (mirrors catalog.models.Product).
from pydantic import BaseModel, Field

class Product(BaseModel):
    id: int = Field(ge=1)
    name: str = Field(min_length=1)
    category: str
    price: float = Field(ge=0)
    tags: list[str] = Field(default_factory=list)

def test_product_defaults():
    p = Product(id=1, name="Cable", category="Electronics", price=499.0)
    assert p.price == 499.0
    assert p.tags == []          # the model's default

test_product_defaults()
print("test_product_defaults: passed")

In [ ]:
# 1.4 · A query — seed a tiny Catalog-like collection (mirrors catalog.models.ProductCatalog).
class CatalogError(Exception):
    """Raised when a catalog operation fails (mirrors catalog.models.CatalogError)."""

class Catalog:
    def __init__(self, products=None):
        self._items = {}
        for p in products or []:
            self.add(p)

    def add(self, product):
        if product.id in self._items:
            raise CatalogError(f"Product id {product.id} already exists")
        self._items[product.id] = product
        return product

    def get(self, product_id):
        if product_id not in self._items:
            raise CatalogError(f"Product id {product_id} not found")
        return self._items[product_id]

    def delete(self, product_id):
        if product_id not in self._items:
            raise CatalogError(f"Product id {product_id} not found")
        return self._items.pop(product_id)

    def list_all(self):
        return list(self._items.values())

    def __len__(self):
        return len(self._items)


def test_catalog_query():
    cat = Catalog([
        Product(id=1, name="Cable", category="Electronics", price=499.0),
        Product(id=2, name="Mat", category="Fitness", price=1299.0),
    ])
    assert len(cat.list_all()) == 2

test_catalog_query()
print("test_catalog_query: passed")

### 1.5 · Discovery and selection

pytest finds files named `test_*.py` and functions named `test_*` — **no registration needed**.
Select a subset with `-k`, see individual names with `-v`:

```bash
pytest -q                     # run everything, dots only
pytest -k search              # run only tests whose name contains "search"
pytest -v                     # print each test's name + PASSED/FAILED
```

### ▶ Terminal moment — the *real* pytest run

The cells above call each `test_*()` by hand and print "passed" — but that's not what pytest
looks like. **Switch to a terminal** and run the real runner:

```bash
uvx pytest -q            # dots for passes, no ceremony
```

Then flip an `assert` to something false and rerun. pytest rewrites the line and shows **both
sides** — `assert 2 == 3` — the introspected diff a bare `AssertionError` can't give you. *That*
is why pytest over `unittest`, and you have to see it in a real run to feel it.

## Section 2 — `pytest.raises` + `@pytest.mark.parametrize`

Good code fails loudly; tests must assert the **failure**, not just the happy path.
`with pytest.raises(Exc, match="..."):` passes only if that exception fires inside the block.
When many cases share one body, `@pytest.mark.parametrize` turns one test into a table of
cases — a *used* decorator, same spirit as `@dataclass` or `@app.get`: you apply it, you never
write one.

In [ ]:
# 2.1a · Before pytest.raises — a plain try/except to see the exception fire.
seeded = Catalog([Product(id=10, name="Cable", category="Electronics", price=499.0)])
dup = Product(id=10, name="dup", category="x", price=1.0)

try:
    seeded.add(dup)
except CatalogError as e:
    print(f"caught expected CatalogError: {e}")

In [ ]:
# 2.1b · The pytest.raises idiom — the test framework's way of asserting a failure fires.
import pytest

def test_add_rejects_duplicate_id():
    cat = Catalog([Product(id=10, name="Cable", category="Electronics", price=499.0)])
    dup = Product(id=10, name="dup", category="x", price=1.0)
    with pytest.raises(CatalogError, match="already exists"):
        cat.add(dup)

test_add_rejects_duplicate_id()
print("test_add_rejects_duplicate_id: passed")

In [ ]:
# 2.2 · Use the NARROWEST exception — a bare Exception would also swallow real bugs.
def test_narrow_vs_broad():
    cat = Catalog([Product(id=10, name="Cable", category="Electronics", price=499.0)])
    dup = Product(id=10, name="dup", category="x", price=1.0)

    # ✗ too broad: a TypeError from a real bug in cat.add would ALSO satisfy this
    with pytest.raises(Exception):
        cat.add(dup)

    # ✓ narrow: only CatalogError satisfies it; anything else still fails the test
    with pytest.raises(CatalogError, match="already exists"):
        cat.add(dup)

test_narrow_vs_broad()
print("test_narrow_vs_broad: passed (but always prefer the narrow form shown second)")

In [ ]:
# 2.3 · A second error path: missing id.
def test_get_missing_raises():
    cat = Catalog([Product(id=10, name="Cable", category="Electronics", price=499.0)])
    with pytest.raises(CatalogError, match="not found"):
        cat.get(999)

test_get_missing_raises()
print("test_get_missing_raises: passed")

In [ ]:
# 2.4 · Model rules raise ValidationError — same idiom, different exception type.
from pydantic import ValidationError

def test_empty_name_rejected():
    with pytest.raises(ValidationError):
        Product(id=1, name="", category="c", price=1.0)

test_empty_name_rejected()
print("test_empty_name_rejected: passed")

### 2.5 · Collapse near-identical tests with `@pytest.mark.parametrize`

This only runs under real pytest collection — **shown here, not executed** in the notebook:

```python
@pytest.mark.parametrize("field,value,msg", [
    ("name",  "", "at least 1 character"),
    ("price", -1, "greater than or equal to 0"),
    ("id",     0, "greater than or equal to 1"),
])
def test_rejects_invalid(field, value, msg):
    base = dict(id=1, name="X", category="c", price=10.0)
    base[field] = value
    with pytest.raises(ValidationError) as exc:
        Product(**base)
    assert msg in str(exc.value)
```

Under `pytest -v` this expands into three named tests, one per row:
`test_rejects_invalid[name--at least 1 character]`, `test_rejects_invalid[price--1-...]`,
`test_rejects_invalid[id-0-...]`. The cell below runs the **equivalent loop** by hand — same
table, same assertions, no decorator needed outside pytest.

In [ ]:
# 2.5 · The equivalent loop — same table as the @parametrize above, run by hand.
cases = [
    ("name",  "", "at least 1 character"),
    ("price", -1, "greater than or equal to 0"),
    ("id",     0, "greater than or equal to 1"),
]

for field, value, msg in cases:
    base = dict(id=1, name="X", category="c", price=10.0)
    base[field] = value
    try:
        Product(**base)
    except ValidationError as exc:
        assert msg in str(exc)
        print(f"test_rejects_invalid[{field}-{value}]: passed")
    else:
        raise AssertionError(f"expected ValidationError for field={field!r}")

## Section 3 — Fixtures + `tmp_path`

Repeated setup becomes a `@pytest.fixture`; a test **asks** for it by naming it as a parameter,
and pytest hands each test its **own fresh copy** (isolation). Here we demo the same idea with
a plain function you call directly — under real pytest it would be `@pytest.fixture`-decorated
and injected automatically. pytest also ships built-in fixtures: `tmp_path` gives a real
temporary directory, the *correct* way to test file I/O.

In [ ]:
# 3.1 · A @pytest.fixture is just a function; call it directly to see the idea.
def seeded_catalog():
    """Under real pytest: @pytest.fixture-decorated, then injected by naming it as a
    test parameter. Here we just call it — same fresh-object-per-call behavior."""
    return Catalog([
        Product(id=10, name="Cable", category="Electronics", price=499.0),
        Product(id=11, name="Speaker", category="Electronics", price=2499.0),
    ])

def test_mutate_one():
    cat = seeded_catalog()
    cat.delete(10)
    assert len(cat) == 1

def test_other_is_unaffected():
    cat = seeded_catalog()          # a FRESH call = a fresh catalog
    assert len(cat) == 2            # the delete above never happened here

test_mutate_one()
test_other_is_unaffected()
print("fixture isolation demo: passed (each call to seeded_catalog() is independent)")

### 3.2 · `conftest.py`

Shared fixtures move into a `conftest.py` file in the tests folder; every test file there can
ask for them **with no import** — pytest auto-discovers `conftest.py`. That's exactly what the
lab's provided `tests/conftest.py` does with `sample_product` and `seeded_catalog`.

In [ ]:
# 3.3 · tmp_path-style round-trip — a REAL temp directory (tempfile stands in for pytest's
# built-in tmp_path fixture), a REAL JSON save + load. This is the correct way to test file I/O.
import json
import tempfile
from pathlib import Path

def save_json(catalog, path):
    path = Path(path)
    payload = [p.model_dump() for p in catalog.list_all()]
    path.write_text(json.dumps(payload, indent=2))

def load_json(path):
    path = Path(path)
    if not path.exists():
        return Catalog()
    raw = json.loads(path.read_text())
    return Catalog([Product(**row) for row in raw])

def test_json_roundtrip():
    cat = Catalog([
        Product(id=1, name="Cable", category="Electronics", price=499.0),
        Product(id=2, name="Mat", category="Fitness", price=1299.0),
    ])
    with tempfile.TemporaryDirectory() as tmp_dir:
        path = Path(tmp_dir) / "catalog.json"
        save_json(cat, path)
        loaded = load_json(path)
        assert loaded.list_all() == cat.list_all()

test_json_roundtrip()
print("test_json_roundtrip: passed")

### 3.4 · Why not mock the filesystem here

The temp file is **cheap** — creating and reading it costs microseconds. Mocking
`Path.write_text` / `read_text` would only prove your code *called* those functions, not that
your JSON actually **round-trips**. Run the real thing when the real thing is cheap; that
judgment is exactly what M08 builds on next, at the one edge that *isn't* cheap: the network.

## Next: the real thing

This notebook showed the ideas on tiny inline stand-ins. In the lab you write the SAME kind of
tests against the real `catalog.models.Product` / `ProductCatalog` and
`catalog.storage.save_json` / `load_json`:

```bash
uv run pytest tests/test_models.py tests/test_catalog.py tests/test_storage.py -q
```
```text
......................
22 passed in 0.02s
```

See `modules/m07-pytest-basics/lab/README.md` for the full lab.